# Dispersion Calculator

Calculate the refractive index at any wavelength for common optical glasses, plot dispersion curves, and compare the Abbe number and partial dispersion of different glass types.

**Physics recap:**
The Sellmeier equation gives the most accurate model for glass dispersion:
$$n^2(\lambda) = 1 + \sum_{i} \frac{B_i \lambda^2}{\lambda^2 - C_i}$$
where $\lambda$ is in micrometres and $B_i$, $C_i$ are glass-specific constants.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'

## 1. Glass Database

In [ ]:
# Sellmeier coefficients for selected Schott glasses.
# Format: {name: {'B': [B1, B2, B3], 'C': [C1, C2, C3]}}
# All C values in µm².
SELLMEIER = {
    'N-BK7':   {'B': [1.03961212, 0.23179234, 1.01046945],
                'C': [6.00069867e-3, 2.00179144e-2, 1.03560653e2]},
    'N-F2':    {'B': [1.34533359, 0.20979017, 0.93397479],
                'C': [9.97743871e-3, 4.70450767e-2, 1.11886764e2]},
    'N-SF11':  {'B': [1.73759695, 0.31395246, 1.18894207],
                'C': [1.31887070e-2, 6.23068142e-2, 1.55236290e2]},
    'N-FK5':   {'B': [0.84433418, 0.34172993, 0.92819083],
                'C': [4.73791365e-3, 1.49296800e-2, 9.72485246e1]},
    'N-BAK4':  {'B': [1.28834642, 0.13272422, 0.94512098],
                'C': [7.79980626e-3, 3.15631177e-2, 1.05965875e2]},
    'N-SF57':  {'B': [1.87543831, 0.37375749, 2.30001797],
                'C': [1.41749518e-2, 6.40509927e-2, 1.77389795e2]},
}

def sellmeier_n(wavelength_nm: float, glass: str) -> float:
    """Return refractive index at wavelength_nm for a given glass."""
    lam_um = wavelength_nm / 1000.0
    lam2 = lam_um ** 2
    coeffs = SELLMEIER[glass]
    n2 = 1.0 + sum(b * lam2 / (lam2 - c) for b, c in zip(coeffs['B'], coeffs['C']))
    return np.sqrt(n2)

print('n(BK7 @ 589 nm):', round(sellmeier_n(589.3, 'N-BK7'), 5))

## 2. Abbe Number Calculator

In [ ]:
def abbe_number(glass: str) -> float:
    """Compute the Abbe number V = (n_d - 1) / (n_F - n_C)."""
    n_d = sellmeier_n(589.3, glass)
    n_F = sellmeier_n(486.1, glass)
    n_C = sellmeier_n(656.3, glass)
    return (n_d - 1.0) / (n_F - n_C)

def partial_dispersion(glass: str) -> float:
    """Partial dispersion ratio P_{g,F}."""
    n_g = sellmeier_n(435.8, glass)
    n_F = sellmeier_n(486.1, glass)
    n_C = sellmeier_n(656.3, glass)
    return (n_g - n_F) / (n_F - n_C)

print(f"{'Glass':<12} {'n_d':>7} {'V':>8} {'P_gF':>8}")
print('-' * 40)
for g in SELLMEIER:
    n_d = sellmeier_n(589.3, g)
    V   = abbe_number(g)
    P   = partial_dispersion(g)
    print(f"{g:<12} {n_d:>7.4f} {V:>8.2f} {P:>8.4f}")

## 3. Dispersion Curves

In [ ]:
wavelengths = np.linspace(380, 750, 400)

COLORS = {
    'N-BK7':   '#4477AA',
    'N-F2':    '#EE6677',
    'N-SF11':  '#AA3377',
    'N-FK5':   '#228833',
    'N-BAK4':  '#CCBB44',
    'N-SF57':  '#66CCEE',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: n(λ) curves
ax = axes[0]
for glass, color in COLORS.items():
    ns = [sellmeier_n(lam, glass) for lam in wavelengths]
    ax.plot(wavelengths, ns, color=color, lw=2, label=glass)

# Reference spectral lines
for lam, label in [(656.3,'C'), (589.3,'d'), (486.1,'F'), (435.8,'g')]:
    ax.axvline(lam, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.text(lam + 2, ax.get_ylim()[1] if ax.get_ylim()[1] > 1 else 1.9,
            label, fontsize=8, color='gray')

ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Refractive index n')
ax.set_title('Dispersion curves n(λ)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: Schott glass map (n_d vs V)
ax2 = axes[1]
for glass, color in COLORS.items():
    n_d = sellmeier_n(589.3, glass)
    V   = abbe_number(glass)
    ax2.scatter(V, n_d, color=color, s=80, zorder=5)
    ax2.annotate(glass, (V, n_d), textcoords='offset points', xytext=(5, 3), fontsize=8)

ax2.invert_xaxis()
ax2.set_xlabel('Abbe number V (decreasing →)')
ax2.set_ylabel('n_d')
ax2.set_title('Glass map (Schott diagram)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Focal Length vs Wavelength

For a thin lens with radii R₁ and R₂ the focal length is:
$$\frac{1}{f(\lambda)} = (n(\lambda) - 1) \left(\frac{1}{R_1} - \frac{1}{R_2}\right)$$

In [ ]:
def focal_length_vs_wavelength(R1, R2, glass, wavelengths_nm):
    """Return focal lengths for a thin lens at given wavelengths."""
    curvature = 1.0 / R1 - 1.0 / R2
    ns = np.array([sellmeier_n(lam, glass) for lam in wavelengths_nm])
    return 1.0 / ((ns - 1.0) * curvature)

# Biconvex lens: R1=+61.5mm, R2=-61.5mm, f_d ≈ 50mm in BK7
R1, R2 = 61.5, -61.5
lams = np.linspace(400, 700, 200)

plt.figure(figsize=(8, 4))
for glass, color in list(COLORS.items())[:4]:
    f = focal_length_vs_wavelength(R1, R2, glass, lams)
    plt.plot(lams, f, color=color, lw=2, label=glass)

plt.xlabel('Wavelength (nm)')
plt.ylabel('Focal length f (mm)')
plt.title('f(λ) — dispersion of focal length for biconvex lens (R₁=+61.5, R₂=−61.5 mm)')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print Abbe-number based CA estimate
print('\nPrimary chromatic aberration Δf = f_d / V:')
f_d = focal_length_vs_wavelength(R1, R2, 'N-BK7', [589.3])[0]
for glass in list(COLORS.keys())[:4]:
    V  = abbe_number(glass)
    fd = focal_length_vs_wavelength(R1, R2, glass, [589.3])[0]
    print(f'  {glass:<12}  f_d={fd:.1f} mm   V={V:.1f}   Δf={fd/V:.3f} mm')